In [ ]:
# # Notebook 04 – CNN Dataset Generation

# ## Objective

# Convert the master demand dataset into CNN-ready tensors.

# Each 30-minute interval is converted into a multi-channel
# spatial tensor that represents the entire city.

# The generated tensors will be used directly by the CNN model
# in Notebook 05.

# ### Inputs

# - demand_master_2026_01.parquet
# - grid_lookup.parquet
# - grid_mask.npy

# ### Outputs

# - X_train.npy
# - y_train.npy
# - X_test.npy
# - y_test.npy
# - metadata.pkl

In [ ]:
# =====================================================
# IMPORT LIBRARIES
# =====================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import geopandas as gpd

from google.colab import drive

import os
import shutil
import time

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import drive
import os
import shutil
import time # Added for small delays

mountpoint = '/content/drive'

# --- Step 1: Attempt to unmount if already mounted ---
if os.path.ismount(mountpoint):
    print(f"Detected {mountpoint} is already mounted. Attempting to unmount...")
    try:
        !fusermount -uz {mountpoint}
        print(f"Successfully unmounted {mountpoint} (if it was a fusermount). This clears any temporary files on the Colab VM's local filesystem at this location, not your actual Google Drive files.")
    except Exception as e:
        print(f"Error during fusermount unmount attempt: {e}. Proceeding with directory cleanup.")
else:
    print(f"{mountpoint} is not currently an active mount point.")

# Add a small delay to ensure unmount operations fully propagate
time.sleep(1)

# --- Step 2: Forcibly remove and recreate the mountpoint directory to ensure it's empty ---
print(f"Ensuring {mountpoint} is a fresh, empty directory before mounting...")
if os.path.exists(mountpoint):
    try:
        shutil.rmtree(mountpoint)
        print(f"Successfully removed existing {mountpoint} directory and its contents from the Colab VM's local filesystem.")
        time.sleep(0.5) # Give a moment for filesystem to register removal
    except OSError as e:
        print(f"Error removing {mountpoint}: {e}. This indicates a persistent lock or permission issue.")
        print("Attempting to create directory, but mount might still fail if directory is locked.")
else:
    print(f"Directory {mountpoint} does not exist on the Colab VM, creating it.")

# Always create the mountpoint directory to ensure it's an empty, clean target
# This ensures it's empty right before drive.mount is called
os.makedirs(mountpoint, exist_ok=True)
print(f"Created/ensured empty mountpoint directory: {mountpoint} on the Colab VM.")
time.sleep(0.5) # Another small delay

# --- Step 3: Attempt to mount Google Drive ---
print("Attempting to mount Google Drive...")
try:
    drive.mount(mountpoint, force_remount=True)
    print("Google Drive mount attempt complete.")
    if os.path.ismount(mountpoint):
        print("Google Drive successfully mounted! Your files should now be accessible via this path.")
    else:
        print("Google Drive did not mount successfully, despite no immediate exception.")
except Exception as e:
    print(f"An error occurred during drive.mount: {e}")
    print("Please check your permissions, restart the runtime, or ensure no other process is using the mountpoint.")

from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras import layers, models

/content/drive is not currently an active mount point.
Ensuring /content/drive is a fresh, empty directory before mounting...
Directory /content/drive does not exist on the Colab VM, creating it.
Created/ensured empty mountpoint directory: /content/drive on the Colab VM.
Attempting to mount Google Drive...
Mounted at /content/drive
Google Drive mount attempt complete.
Google Drive successfully mounted! Your files should now be accessible via this path.


In [ ]:
# =====================================================
# PROJECT PATHS
# =====================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/RideHailingPhD")

PROCESSED_DIR = PROJECT_ROOT / "processed_data"

SPATIAL_DIR = PROCESSED_DIR / "spatial"
DEMAND_DIR = PROCESSED_DIR / "demand"

CNN_DIR = PROCESSED_DIR / "cnn"

CNN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
# =====================================================
# CONFIGURATION
# =====================================================

GRID_ROWS = 51
GRID_COLS = 52

TRAIN_RATIO = 0.80

USE_CURRENT_DEMAND = False

BASE_CHANNELS = [

    "HistoricalDemand",

    "Temperature",

    "WindSpeed",

    "WeatherCode",

    "DayOfWeek",

    "Hour",

    "Period"

]

if USE_CURRENT_DEMAND:

    CHANNELS = ["Demand"] + BASE_CHANNELS

else:

    CHANNELS = BASE_CHANNELS

print("CNN Channels")

for i, channel in enumerate(CHANNELS, start=1):

    print(f"{i}. {channel}")

print()

print(f"Total Channels : {len(CHANNELS)}")

CNN Channels
1. HistoricalDemand
2. Temperature
3. WindSpeed
4. WeatherCode
5. DayOfWeek
6. Hour
7. Period

Total Channels : 7


In [ ]:
# =====================================================
# LOAD DATASETS
# =====================================================

master_demand = pd.read_parquet(
    DEMAND_DIR /
    "demand_master_2026_01.parquet"
)

grid_lookup = pd.read_parquet(
    SPATIAL_DIR /
    "grid_lookup.parquet"
)

grid_mask = np.load(
    SPATIAL_DIR /
    "grid_mask.npy"
)

print(f"Master Dataset : {len(master_demand):,}")

print(f"Grid Lookup    : {len(grid_lookup):,}")

print(f"Mask Shape     : {grid_mask.shape}")

display(master_demand.head())

Master Dataset : 1,804,944
Grid Lookup    : 1,213
Mask Shape     : (51, 52)


,TimeSlot,GridID,Demand,Row,Column,Temperature,WindSpeed,WeatherCode,DayOfWeek,Hour,Minute,Period,TimeIndex,HistoricalDemand
0,2026-01-01,0,0,0,31,-0.0,17.0,1.0,3,0,0,0,0,0.0
1,2026-01-01,1,0,0,32,-0.0,17.0,1.0,3,0,0,0,0,0.0
2,2026-01-01,2,0,0,33,-0.0,17.0,1.0,3,0,0,0,0,0.0
3,2026-01-01,3,0,0,34,-0.0,17.0,1.0,3,0,0,0,0,0.0
4,2026-01-01,4,0,0,37,-0.0,17.0,1.0,3,0,0,0,0,0.0


In [ ]:
# =====================================================
# VALIDATE INPUTS
# =====================================================

print("=" * 60)

print("INPUT SUMMARY")

print("=" * 60)

print(f"Time Slots : {master_demand['TimeIndex'].nunique()}")

print(f"Grid Cells : {master_demand['GridID'].nunique()}")

print(f"Rows       : {len(master_demand):,}")

print()

print("Channels")

for channel in CHANNELS:

    assert channel in master_demand.columns

    print(channel)

assert grid_mask.shape == (GRID_ROWS, GRID_COLS)

assert master_demand["GridID"].nunique() == len(grid_lookup)

print()

print("Input validation passed.")

INPUT SUMMARY
Time Slots : 1488
Grid Cells : 1213
Rows       : 1,804,944

Channels
HistoricalDemand
Temperature
WindSpeed
WeatherCode
DayOfWeek
Hour
Period

Input validation passed.


In [ ]:
# # =====================================================
# # STAGE 2 : BUILD FEATURE TENSOR
# # =====================================================

# ## Objective

# Convert one 30-minute demand interval into a multi-channel
# CNN input tensor.

# Each feature is converted into a 51 × 52 spatial map.

# The resulting tensor has shape:

# (Number of Channels, Height, Width)

# For the paper implementation:

# (7, 51, 52)

In [ ]:
# =====================================================
# SELECT ONE TIME STEP
# =====================================================

TIME_INDEX = 0

snapshot = (
    master_demand[
        master_demand["TimeIndex"] == TIME_INDEX
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Time Index : {TIME_INDEX}")
print(f"Rows       : {len(snapshot)}")

display(snapshot.head())

Time Index : 0
Rows       : 1213


,TimeSlot,GridID,Demand,Row,Column,Temperature,WindSpeed,WeatherCode,DayOfWeek,Hour,Minute,Period,TimeIndex,HistoricalDemand
0,2026-01-01,0,0,0,31,-0.0,17.0,1.0,3,0,0,0,0,0.0
1,2026-01-01,1,0,0,32,-0.0,17.0,1.0,3,0,0,0,0,0.0
2,2026-01-01,2,0,0,33,-0.0,17.0,1.0,3,0,0,0,0,0.0
3,2026-01-01,3,0,0,34,-0.0,17.0,1.0,3,0,0,0,0,0.0
4,2026-01-01,4,0,0,37,-0.0,17.0,1.0,3,0,0,0,0,0.0


In [ ]:
# =====================================================
# BUILD FEATURE MAP
# =====================================================

def build_feature_map(df, feature_name, rows, cols):
    """
    Convert one feature into a 2D spatial map.
    """

    feature_map = np.zeros(
        (rows, cols),
        dtype=np.float32
    )

    feature_map[
        df["Row"].to_numpy(),
        df["Column"].to_numpy()
    ] = df[feature_name].to_numpy(dtype=np.float32)

    return feature_map

In [ ]:
# =====================================================
# GENERATE FEATURE MAPS
# =====================================================

feature_maps = []

for feature in CHANNELS:

    fmap = build_feature_map(
        snapshot,
        feature,
        GRID_ROWS,
        GRID_COLS
    )

    feature_maps.append(fmap)

print(f"Feature Maps : {len(feature_maps)}")

Feature Maps : 7


In [ ]:
# =====================================================
# CREATE CNN TENSOR
# =====================================================

cnn_tensor = np.stack(
    feature_maps,
    axis=0
)

print("Tensor Shape :", cnn_tensor.shape)

assert cnn_tensor.shape == (
    len(CHANNELS),
    GRID_ROWS,
    GRID_COLS
)

print("\nTensor validation passed.")

Tensor Shape : (7, 51, 52)

Tensor validation passed.


In [ ]:
# =====================================================
# INSPECT FEATURE MAPS (VALID GRID CELLS ONLY)
# =====================================================

valid_mask = grid_mask.astype(bool)

for i, feature in enumerate(CHANNELS):

    fmap = cnn_tensor[i]

    values = fmap[valid_mask]

    print("=" * 60)
    print(feature)
    print("=" * 60)

    print(f"Shape : {fmap.shape}")
    print(f"Min   : {values.min():.3f}")
    print(f"Max   : {values.max():.3f}")
    print(f"Mean  : {values.mean():.3f}")

    print()

HistoricalDemand
Shape : (51, 52)
Min   : 0.000
Max   : 0.000
Mean  : 0.000

Temperature
Shape : (51, 52)
Min   : -0.000
Max   : -0.000
Mean  : 0.000

WindSpeed
Shape : (51, 52)
Min   : 17.000
Max   : 17.000
Mean  : 17.000

WeatherCode
Shape : (51, 52)
Min   : 1.000
Max   : 1.000
Mean  : 1.000

DayOfWeek
Shape : (51, 52)
Min   : 3.000
Max   : 3.000
Mean  : 3.000

Hour
Shape : (51, 52)
Min   : 0.000
Max   : 0.000
Mean  : 0.000

Period
Shape : (51, 52)
Min   : 0.000
Max   : 0.000
Mean  : 0.000



In [ ]:
snapshot[
    [
        "Temperature",
        "WindSpeed",
        "WeatherCode"
    ]
].drop_duplicates()

,Temperature,WindSpeed,WeatherCode
0,-0.0,17.0,1.0


In [ ]:
# =====================================================
# BUILD TIME SNAPSHOT LOOKUP
# =====================================================

snapshots = {
    time_index: group.reset_index(drop=True)
    for time_index, group in master_demand.groupby("TimeIndex")
}

print(f"Snapshots Created : {len(snapshots)}")

first_snapshot = snapshots[0]

print(f"Rows in Snapshot 0 : {len(first_snapshot)}")

Snapshots Created : 1488
Rows in Snapshot 0 : 1213


In [ ]:
# =====================================================
# BUILD CNN TENSOR
# =====================================================

def build_tensor(snapshot):

    feature_maps = []

    for feature in CHANNELS:

        fmap = build_feature_map(
            snapshot,
            feature,
            GRID_ROWS,
            GRID_COLS
        )

        feature_maps.append(fmap)

    return np.stack(
        feature_maps,
        axis=0
    )

In [ ]:
# =====================================================
# BUILD TARGET MAP
# =====================================================

def build_target(snapshot):

    target = build_feature_map(
        snapshot,
        "Demand",
        GRID_ROWS,
        GRID_COLS
    )

    return target[np.newaxis, :, :]

In [ ]:
# =====================================================
# GENERATE CNN DATASET
# =====================================================

X = []

y = []

time_slots = []

unique_times = sorted(snapshots.keys())

print(f"Generating {len(unique_times)-1:,} samples...")

for t in unique_times[:-1]:

    current = snapshots[t]

    future = snapshots[t + 1]

    X.append(
        build_tensor(current)
    )

    y.append(
        build_target(future)
    )

    time_slots.append(
        current["TimeSlot"].iloc[0]
    )

X = np.asarray(
    X,
    dtype=np.float32
)

y = np.asarray(
    y,
    dtype=np.float32
)

time_slots = np.asarray(time_slots)

print()

print("Dataset generation completed.")

Generating 1,487 samples...

Dataset generation completed.


In [ ]:
# =====================================================
# DATASET SUMMARY
# =====================================================

print("=" * 70)
print("CNN DATASET SUMMARY")
print("=" * 70)

print(f"Samples          : {len(X):,}")
print(f"Channels         : {X.shape[1]}")
print(f"Height           : {X.shape[2]}")
print(f"Width            : {X.shape[3]}")

print()

print(f"Target Samples   : {len(y):,}")

print(f"Target Shape     : {y.shape[1:]}")

print()

print(f"First Time Slot  : {time_slots[0]}")
print(f"Last Time Slot   : {time_slots[-1]}")

print()

print("Input Shape")

print(X.shape)

print()

print("Target Shape")

print(y.shape)

CNN DATASET SUMMARY
Samples          : 1,487
Channels         : 7
Height           : 51
Width            : 52

Target Samples   : 1,487
Target Shape     : (1, 51, 52)

First Time Slot  : 2026-01-01 00:00:00
Last Time Slot   : 2026-01-31 23:00:00

Input Shape
(1487, 7, 51, 52)

Target Shape
(1487, 1, 51, 52)


In [ ]:
# =====================================================
# VALIDATE SAMPLE
# =====================================================

sample = 0

print("Input Tensor")

print(X[sample].shape)

print()

print("Target Tensor")

print(y[sample].shape)

assert X.shape[0] == y.shape[0]

assert X.shape[1] == len(CHANNELS)

assert y.shape[1] == 1

assert X.shape[2:] == (
    GRID_ROWS,
    GRID_COLS
)

assert y.shape[2:] == (
    GRID_ROWS,
    GRID_COLS
)

print()

print("CNN dataset validation passed.")

Input Tensor
(7, 51, 52)

Target Tensor
(1, 51, 52)

CNN dataset validation passed.


In [ ]:
# =====================================================
# TRAIN / TEST SPLIT
# =====================================================

split_index = int(
    len(X) * TRAIN_RATIO
)

X_train = X[:split_index]

X_test = X[split_index:]

y_train = y[:split_index]

y_test = y[split_index:]

time_train = time_slots[:split_index]

time_test = time_slots[split_index:]

In [ ]:
# =====================================================
# SPLIT SUMMARY
# =====================================================

print("=" * 70)
print("TRAIN / TEST SUMMARY")
print("=" * 70)

print(f"Training Samples : {len(X_train):,}")
print(f"Testing Samples  : {len(X_test):,}")

print()

print("Training Shape")

print(X_train.shape)

print(y_train.shape)

print()

print("Testing Shape")

print(X_test.shape)

print(y_test.shape)

print()

print("Training Period")

print(time_train[0])

print("to")

print(time_train[-1])

print()

print("Testing Period")

print(time_test[0])

print("to")

print(time_test[-1])

TRAIN / TEST SUMMARY
Training Samples : 1,189
Testing Samples  : 298

Training Shape
(1189, 7, 51, 52)
(1189, 1, 51, 52)

Testing Shape
(298, 7, 51, 52)
(298, 1, 51, 52)

Training Period
2026-01-01 00:00:00
to
2026-01-25 18:00:00

Testing Period
2026-01-25 18:30:00
to
2026-01-31 23:00:00


In [ ]:
# =====================================================
# SAVE CNN DATASET
# =====================================================

np.save(
    CNN_DIR / "X_train.npy",
    X_train
)

np.save(
    CNN_DIR / "y_train.npy",
    y_train
)

np.save(
    CNN_DIR / "X_test.npy",
    X_test
)

np.save(
    CNN_DIR / "y_test.npy",
    y_test
)

np.save(
    CNN_DIR / "time_train.npy",
    time_train
)

np.save(
    CNN_DIR / "time_test.npy",
    time_test
)

In [ ]:
# =====================================================
# SAVE METADATA
# =====================================================

metadata = {

    "rows": GRID_ROWS,

    "cols": GRID_COLS,

    "channels": CHANNELS,

    "train_samples": len(X_train),

    "test_samples": len(X_test),

    "train_ratio": TRAIN_RATIO,

    "input_shape": X_train.shape,

    "target_shape": y_train.shape

}

with open(
    CNN_DIR / "metadata.pkl",
    "wb"
) as f:

    pickle.dump(
        metadata,
        f
    )

print("Metadata saved.")

Metadata saved.


In [ ]:
# =====================================================
# FINAL SUMMARY
# =====================================================

print("=" * 70)
print("CNN DATASET CREATED")
print("=" * 70)

print()

print("Saved Files")

for file in sorted(CNN_DIR.iterdir()):

    print(file.name)

print()

print("Input Shape")

print(X_train.shape)

print()

print("Target Shape")

print(y_train.shape)

print()

print("NB4 completed successfully.")

CNN DATASET CREATED

Saved Files
X_test.npy
X_train.npy
metadata.pkl
time_test.npy
time_train.npy
y_test.npy
y_train.npy

Input Shape
(1189, 7, 51, 52)

Target Shape
(1189, 1, 51, 52)

NB4 completed successfully.
